# 06 Improve WLASL1000 — BiGRU + Attention Light V2

## Purpose
This notebook improves the WLASL1000 BiGRU model without making it too heavy.

## Baseline result
Current WLASL1000 V1:

```text
Test Top-1 Accuracy: 20.56%
Test Top-3 Accuracy: 38.68%
Test Top-5 Accuracy: 46.29%
Test Macro F1: 16.74%
```

## Strategy
This notebook keeps the successful V1 approach but adds safe improvements:

- keypoints + velocity only
- light temporal augmentation
- slightly larger hidden size
- tuned dropout
- same attention structure
- V1 comparison and confidence threshold analysis

In [1]:
from pathlib import Path
import json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

e:\Be_My_Ear\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup paths and Light V2 configuration

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Clean index:", CLEAN_INDEX_FILE.exists())

MODEL_NAME = "bigru_attention_light_v2"
MODEL_DISPLAY_NAME = "BiGRU + Attention Light V2"
TRAINING_TITLE = "Be My Ear - WLASL1000 BiGRU + Attention Light V2"

MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}.pt"
HISTORY_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_history.csv"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_light_v2_train_norm_stats.npz"
RESULT_FILE = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"

BATCH_SIZE = 16
EPOCHS = 80
EARLY_STOPPING_PATIENCE = 16

USE_VELOCITY = True
INPUT_SIZE = 516
SEQUENCE_LENGTH = 60

HIDDEN_SIZE = 320
NUM_LAYERS = 2
DROPOUT = 0.35

print("Model path:", MODEL_PATH)

Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Clean index: True
Model path: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_light_v2_wlasl1000.pt


## 2. Load clean dataset and split data

In [3]:
df = pd.read_csv(CLEAN_INDEX_FILE)

train_records, val_records, test_records = [], [], []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)
    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_records.append(group.iloc[:n_test])
    val_records.append(group.iloc[n_test:n_test + n_val])
    train_records.append(group.iloc[n_test + n_val:])

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

NUM_CLASSES = df["label_id"].nunique()

print("Clean samples:", len(df))
print("Classes:", NUM_CLASSES)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Clean samples: 7232
Classes: 1000
Train: 5024 Val: 1104 Test: 1104


## 3. Compute normalisation using train set only

In [4]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)
    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved norm stats:", NORM_STATS_PATH)

Computing train mean/std: 100%|██████████| 5024/5024 [00:00<00:00, 5984.01it/s]

Saved norm stats: E:\Be_My_Ear\models\ASL\WLASL1000\wlasl1000_light_v2_train_norm_stats.npz


## 4. Dataset with light temporal augmentation

In [5]:
class WLASLKeypointDatasetLightAug(Dataset):
    def __init__(self, dataframe, mean, std, augment=False, noise_std=0.005, frame_mask_prob=0.03, temporal_shift_max=2):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.augment = augment
        self.noise_std = noise_std
        self.frame_mask_prob = frame_mask_prob
        self.temporal_shift_max = temporal_shift_max

    def __len__(self):
        return len(self.dataframe)

    def _temporal_shift(self, x):
        shift = np.random.randint(-self.temporal_shift_max, self.temporal_shift_max + 1)
        if shift == 0:
            return x
        shifted = np.zeros_like(x)
        if shift > 0:
            shifted[shift:] = x[:-shift]
            shifted[:shift] = x[0]
        else:
            shifted[:shift] = x[-shift:]
            shifted[shift:] = x[-1]
        return shifted

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.augment:
            keypoints = self._temporal_shift(keypoints)

        velocity = np.zeros_like(keypoints, dtype=np.float32)
        velocity[1:] = keypoints[1:] - keypoints[:-1]
        features = np.concatenate([keypoints, velocity], axis=1).astype(np.float32)

        if self.augment:
            features += np.random.normal(0, self.noise_std, features.shape).astype(np.float32)
            mask = np.random.rand(features.shape[0]) < self.frame_mask_prob
            features[mask] = 0

        label = int(row["label_id"])
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

## 5. Create balanced data loaders

In [6]:
train_dataset = WLASLKeypointDatasetLightAug(train_df, train_mean, train_std, augment=True)
val_dataset = WLASLKeypointDatasetLightAug(val_df, train_mean, train_std, augment=False)
test_dataset = WLASLKeypointDatasetLightAug(test_df, train_mean, train_std, augment=False)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values

sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("Input batch shape:", x_batch.shape)

Input batch shape: torch.Size([16, 60, 516])


## 6. Define BiGRU + Attention Light V2 model

In [7]:
class BiGRUAttentionLightV2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.gru = nn.GRU(
            hidden_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        bi_hidden = hidden_size * 2
        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)

def build_model_from_checkpoint(checkpoint):
    return BiGRUAttentionLightV2(
        input_size=checkpoint.get("input_size", INPUT_SIZE),
        hidden_size=checkpoint.get("hidden_size", HIDDEN_SIZE),
        num_classes=checkpoint.get("num_classes", NUM_CLASSES),
        num_layers=checkpoint.get("num_layers", NUM_LAYERS),
        dropout=checkpoint.get("dropout", DROPOUT)
    )

## 7. Initialise Light V2 model

In [8]:
model = BiGRUAttentionLightV2(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES, NUM_LAYERS, DROPOUT).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

print("Parameters:", sum(p.numel() for p in model.parameters()))

def create_checkpoint_payload(epoch, best_val_f1, best_val_top5):
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_f1": best_val_f1,
        "best_val_top5": best_val_top5,
        "num_classes": NUM_CLASSES,
        "input_size": INPUT_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "use_velocity": USE_VELOCITY,
        "architecture": "BiGRUAttentionLightV2",
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT
    }

Parameters: 3977961


## 8. Training helper functions

In [9]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    return top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds)).any(dim=1).float().mean().item()

def run_epoch(model, loader, criterion, optimizer=None, phase="Train", epoch=1, total_epochs=1):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = total_top1 = total_top3 = total_top5 = 0
    all_preds, all_labels = [], []

    bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(bar, start=1):
            x, y = x.to(device), y.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item()
            total_top1 += (preds == y).float().mean().item()
            total_top3 += top_k_accuracy(outputs, y, 3)
            total_top5 += top_k_accuracy(outputs, y, 5)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

            bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{(preds == y).float().mean().item():.4f}",
                "top5": f"{top_k_accuracy(outputs, y, 5):.4f}"
            })

    return (
        total_loss / len(loader),
        total_top1 / len(loader),
        total_top3 / len(loader),
        total_top5 / len(loader),
        f1_score(all_labels, all_preds, average="macro", zero_division=0)
    )

def top_k_accuracy_numpy(y_true, y_probs, k):
    correct = 0
    for true_label, prob in zip(y_true, y_probs):
        if true_label in np.argsort(prob)[-k:]:
            correct += 1
    return correct / len(y_true)

def collect_predictions(model, loader):
    model.eval()
    labels, preds, probs_all = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            outputs = model(x)
            probs = F.softmax(outputs, dim=1)
            pred = torch.argmax(probs, dim=1)

            labels.extend(y.numpy())
            preds.extend(pred.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())

    return np.array(labels), np.array(preds), np.array(probs_all)

## 9. Train Light V2 model

In [10]:
history = {k: [] for k in [
    "train_loss", "train_top1", "train_top3", "train_top5", "train_f1",
    "val_loss", "val_top1", "val_top3", "val_top5", "val_f1", "lr"
]}

best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print(TRAINING_TITLE)
print("=" * 80)
print("Device:", device)
print("Input shape:", (60, INPUT_SIZE))
print("Classes:", NUM_CLASSES)
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Model path:", MODEL_PATH)
print("=" * 80)

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_loss, train_top1, train_top3, train_top5, train_f1 = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, phase="Training", epoch=epoch, total_epochs=EPOCHS
    )

    val_loss, val_top1, val_top3, val_top5, val_f1 = run_epoch(
        model, val_loader, criterion, optimizer=None, phase="Validation", epoch=epoch, total_epochs=EPOCHS
    )

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_top1"].append(train_top1)
    history["train_top3"].append(train_top3)
    history["train_top5"].append(train_top5)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top3"].append(val_top3)
    history["val_top5"].append(val_top5)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        payload = create_checkpoint_payload(epoch, best_val_f1, best_val_top5)
        torch.save(payload, MODEL_PATH)
        save_status = "Saved new best model"
    else:
        epochs_without_improvement += 1
        save_status = "No improvement"

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")
    print(f"Learning rate: {current_lr:.8f}")
    print("Status:", save_status)
    print(f"Best Val F1 so far: {best_val_f1:.4f}")
    print(f"Best Val Top-5 so far: {best_val_top5:.4f}")
    print(f"Epochs without improvement: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nTraining completed in minutes:", round((time.time() - start_time) / 60, 2))
print("Best model saved:", MODEL_PATH)

Be My Ear - WLASL1000 BiGRU + Attention Light V2
Device: cuda
Input shape: (60, 516)
Classes: 1000
Train samples: 5024
Validation samples: 1104
Test samples: 1104
Epochs: 80
Batch size: 16
Model path: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_light_v2_wlasl1000.pt

Epoch 1/80
--------------------------------------------------------------------------------


Train | Loss: 6.8069 | Top-1: 0.0070 | Top-3: 0.0169 | Top-5: 0.0249 | F1: 0.0022
Val   | Loss: 6.6339 | Top-1: 0.0082 | Top-3: 0.0154 | Top-5: 0.0236 | F1: 0.0017
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0017
Best Val Top-5 so far: 0.0236
Epochs without improvement: 0/16

Epoch 2/80
--------------------------------------------------------------------------------


Train | Loss: 6.3074 | Top-1: 0.0241 | Top-3: 0.0539 | Top-5: 0.0762 | F1: 0.0081
Val   | Loss: 6.2981 | Top-1: 0.0145 | Top-3: 0.0389 | Top-5: 0.0580 | F1: 0.0042
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0042
Best Val Top-5 so far: 0.0580
Epochs without improvement: 0/16

Epoch 3/80
--------------------------------------------------------------------------------


Train | Loss: 5.8776 | Top-1: 0.0486 | Top-3: 0.0959 | Top-5: 0.1270 | F1: 0.0198
Val   | Loss: 5.9780 | Top-1: 0.0317 | Top-3: 0.0607 | Top-5: 0.0861 | F1: 0.0104
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0104
Best Val Top-5 so far: 0.0861
Epochs without improvement: 0/16

Epoch 4/80
--------------------------------------------------------------------------------


Train | Loss: 5.4888 | Top-1: 0.0727 | Top-3: 0.1507 | Top-5: 0.2020 | F1: 0.0395
Val   | Loss: 5.7500 | Top-1: 0.0308 | Top-3: 0.0743 | Top-5: 0.1205 | F1: 0.0125
Learning rate: 0.00020000
Status: Saved new best model
Best Val F1 so far: 0.0125
Best Val Top-5 so far: 0.1205
Epochs without improvement: 0/16

Epoch 5/80
--------------------------------------------------------------------------------


...

Train | Loss: 0.7156 | Top-1: 0.9747 | Top-3: 1.0000 | Top-5: 1.0000 | F1: 0.9739
Val   | Loss: 4.2901 | Top-1: 0.2464 | Top-3: 0.4393 | Top-5: 0.5163 | F1: 0.1930
Learning rate: 0.00000625
Status: No improvement
Best Val F1 so far: 0.1972
Best Val Top-5 so far: 0.5190
Epochs without improvement: 12/16

Epoch 75/80
--------------------------------------------------------------------------------


Train | Loss: 0.7117 | Top-1: 0.9769 | Top-3: 1.0000 | Top-5: 1.0000 | F1: 0.9754
Val   | Loss: 4.2892 | Top-1: 0.2428 | Top-3: 0.4393 | Top-5: 0.5163 | F1: 0.1887
Learning rate: 0.00000625
Status: No improvement
Best Val F1 so far: 0.1972
Best Val Top-5 so far: 0.5190
Epochs without improvement: 13/16

Epoch 76/80
--------------------------------------------------------------------------------


Train | Loss: 0.7123 | Top-1: 0.9785 | Top-3: 1.0000 | Top-5: 1.0000 | F1: 0.9746
Val   | Loss: 4.2894 | Top-1: 0.2391 | Top-3: 0.4357 | Top-5: 0.5154 | F1: 0.1880
Learning rate: 0.00000625
Status: No improvement
Best Val F1 so far: 0.1972
Best Val Top-5 so far: 0.5190
Epochs without improvement: 14/16

Epoch 77/80
--------------------------------------------------------------------------------


Train | Loss: 0.7160 | Top-1: 0.9747 | Top-3: 1.0000 | Top-5: 1.0000 | F1: 0.9738
Val   | Loss: 4.2914 | Top-1: 0.2418 | Top-3: 0.4393 | Top-5: 0.5127 | F1: 0.1891
Learning rate: 0.00000625
Status: No improvement
Best Val F1 so far: 0.1972
Best Val Top-5 so far: 0.5190
Epochs without improvement: 15/16

Epoch 78/80
--------------------------------------------------------------------------------


Train | Loss: 0.7079 | Top-1: 0.9787 | Top-3: 1.0000 | Top-5: 1.0000 | F1: 0.9771
Val   | Loss: 4.2941 | Top-1: 0.2428 | Top-3: 0.4357 | Top-5: 0.5136 | F1: 0.1902
Learning rate: 0.00000625
Status: No improvement
Best Val F1 so far: 0.1972
Best Val Top-5 so far: 0.5190
Epochs without improvement: 16/16

Early stopping triggered.

Training completed in minutes: 17.28
Best model saved: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_light_v2_wlasl1000.pt


## 10. Save history and evaluate Light V2

In [11]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
print("Saved history:", HISTORY_PATH)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model = build_model_from_checkpoint(checkpoint).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred, y_probs = collect_predictions(model, test_loader)

test_top1 = accuracy_score(y_true, y_pred)
test_top3 = top_k_accuracy_numpy(y_true, y_probs, 3)
test_top5 = top_k_accuracy_numpy(y_true, y_probs, 5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print(f"{MODEL_DISPLAY_NAME} Test Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": MODEL_DISPLAY_NAME,
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "input_shape": f"(60, {INPUT_SIZE})",
    "features": "keypoints + velocity",
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH),
    "norm_stats_path": str(NORM_STATS_PATH)
}])

result_df.to_csv(RESULT_FILE, index=False)
report_result_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"
result_df.to_csv(report_result_file, index=False)

print("Saved result summary:", RESULT_FILE)
display(result_df)

Saved history: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_light_v2_wlasl1000_history.csv


BiGRU + Attention Light V2 Test Evaluation
Test Top-1 Accuracy: 0.2437
Test Top-3 Accuracy: 0.4348
Test Top-5 Accuracy: 0.5109
Test Macro F1: 0.1982
Saved result summary: E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attention_light_v2_wlasl1000_result_summary.csv


,dataset,model,clean_samples,classes,input_shape,features,best_val_f1,best_val_top5,checkpoint_epoch,test_top1_accuracy,test_top3_accuracy,test_top5_accuracy,test_macro_f1,model_path,history_path,norm_stats_path
0,WLASL1000,BiGRU + Attention Light V2,7232,1000,"(60, 516)",keypoints + velocity,0.197183,0.519022,62,0.243659,0.434783,0.51087,0.198214,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...,E:\Be_My_Ear\models\ASL\WLASL1000\wlasl1000_li...


## 11. Confidence threshold analysis

In [12]:
if LABEL_MAP_FILE.exists():
    with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
        label_map = json.load(f)
    id_to_gloss = {int(k): v["gloss"] for k, v in label_map.items()}
else:
    id_to_gloss = {}

test_df_reset = test_df.reset_index(drop=True)
prediction_records = []

for i in range(len(y_true)):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])
    top5_ids = np.argsort(y_probs[i])[-5:][::-1]

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": true_id in top5_ids,
        "top5_glosses": ", ".join([id_to_gloss.get(int(x), str(x)) for x in top5_ids])
    })

predictions_df = pd.DataFrame(prediction_records)
predictions_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_test_predictions.csv"
predictions_df.to_csv(predictions_file, index=False)

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_records = []

for threshold in thresholds:
    confident = predictions_df[predictions_df["confidence"] >= threshold]
    threshold_records.append({
        "confidence_threshold": threshold,
        "coverage": len(confident) / len(predictions_df),
        "top1_accuracy_on_confident_samples": confident["correct_top1"].mean() if len(confident) else np.nan,
        "top5_accuracy_on_confident_samples": confident["correct_top5"].mean() if len(confident) else np.nan,
        "num_confident_samples": len(confident)
    })

threshold_df = pd.DataFrame(threshold_records)
threshold_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_file, index=False)

print("Saved predictions:", predictions_file)
print("Saved confidence threshold analysis:", threshold_file)
display(threshold_df)

Saved predictions: E:\Be_My_Ear\reports\phase1_wlasl1000\bigru_attention_light_v2_wlasl1000_test_predictions.csv
Saved confidence threshold analysis: E:\Be_My_Ear\reports\phase1_wlasl1000\bigru_attention_light_v2_wlasl1000_confidence_threshold_analysis.csv


,confidence_threshold,coverage,top1_accuracy_on_confident_samples,top5_accuracy_on_confident_samples,num_confident_samples
0,0.2,0.561594,0.325806,0.622581,620
1,0.3,0.394022,0.363218,0.671264,435
2,0.4,0.299819,0.380665,0.688822,331
3,0.5,0.232790,0.396887,0.696498,257
4,0.6,0.177536,0.454082,0.734694,196
5,0.7,0.127717,0.446809,0.744681,141
6,0.8,0.093297,0.456311,0.747573,103
7,0.9,0.066123,0.369863,0.698630,73


## 12. Compare V1 vs Light V2

In [13]:
v1_file = MODEL_DIR / f"bigru_attention_{PREFIX}_result_summary.csv"
rows = []

if v1_file.exists():
    v1 = pd.read_csv(v1_file).iloc[0].to_dict()
    rows.append({
        "version": "V1",
        "model": v1.get("model", "BiGRU + Temporal Attention"),
        "test_top1_accuracy": float(v1.get("test_top1_accuracy", np.nan)),
        "test_top3_accuracy": float(v1.get("test_top3_accuracy", np.nan)),
        "test_top5_accuracy": float(v1.get("test_top5_accuracy", np.nan)),
        "test_macro_f1": float(v1.get("test_macro_f1", np.nan)),
        "best_val_f1": float(v1.get("best_val_f1", np.nan)),
        "best_val_top5": float(v1.get("best_val_top5", np.nan)),
        "source": str(v1_file)
    })

rows.append({
    "version": "Light V2",
    "model": MODEL_DISPLAY_NAME,
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "source": str(RESULT_FILE)
})

comparison_df = pd.DataFrame(rows)
comparison_file = REPORT_DIR / f"{PREFIX}_v1_vs_light_v2_comparison.csv"
comparison_df.to_csv(comparison_file, index=False)

print("Saved comparison:", comparison_file)
display(comparison_df)

Saved comparison: E:\Be_My_Ear\reports\phase1_wlasl1000\wlasl1000_v1_vs_light_v2_comparison.csv


,version,model,test_top1_accuracy,test_top3_accuracy,test_top5_accuracy,test_macro_f1,best_val_f1,best_val_top5,source
0,V1,BiGRU + Temporal Attention,0.208036,0.388393,0.463393,0.167371,0.170589,0.463393,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...
1,Light V2,BiGRU + Attention Light V2,0.243659,0.434783,0.510870,0.198214,0.197183,0.519022,E:\Be_My_Ear\models\ASL\WLASL1000\bigru_attent...
